<a href="https://colab.research.google.com/github/sharmin-iffat/Plant-Leaf-Disease-Classification/blob/main/DiseaseClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision matplotlib


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.optim as optim


In [ ]:
train_dir = "/content/drive/MyDrive/Soybean/Train"
test_dir = "/content/drive/MyDrive/Soybean/Test"

# Data augmentation & normalization
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder(train_dir, transform=train_transforms)
test_data = datasets.ImageFolder(test_dir, transform=test_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

num_classes = len(train_data.classes)
print("Classes:", train_data.classes)


Classes: ['Bacterial Pustule', 'Frogeye Leaf Spot', 'Healty', 'Mossaic Virus', 'Rust', 'Southern blight', 'Sudden Death Syndrome', 'Target Leaf Spot', 'Yellow Mosaic', 'brown_spot', 'crestamento', 'ferrugen', 'powdery_mildew', 'septoria']


In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze feature layers
for param in model.features.parameters():
    param.requires_grad = False

# Change classifier layer to match your disease classes
model.classifier[1] = nn.Linear(1280, num_classes)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)


In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # 🔥 PRINT EPOCH RESULT HERE
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")


Epoch [1/10] - Loss: 1.9275
Epoch [2/10] - Loss: 1.1512
Epoch [3/10] - Loss: 0.8981
Epoch [4/10] - Loss: 0.7531
Epoch [5/10] - Loss: 0.7061
Epoch [6/10] - Loss: 0.6299
Epoch [7/10] - Loss: 0.5760
Epoch [8/10] - Loss: 0.5576
Epoch [9/10] - Loss: 0.5212
Epoch [10/10] - Loss: 0.5146


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", (correct / total) * 100)


Test Accuracy: 6.302521008403361


In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/soybean_model.pth")



In [ ]:
from torchvision import datasets
train_data = datasets.ImageFolder("/content/drive/MyDrive/Soybean/Train")
print(train_data.classes)


['Bacterial Pustule', 'Frogeye Leaf Spot', 'Healty', 'Mossaic Virus', 'Rust', 'Southern blight', 'Sudden Death Syndrome', 'Target Leaf Spot', 'Yellow Mosaic', 'brown_spot', 'crestamento', 'ferrugen', 'powdery_mildew', 'septoria']


In [ ]:
!pip install gradio


In [ ]:
import gradio as gr
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image

# --------------------------
# 1. Define all classes
# --------------------------
classes = ['Bacterial Pustule', 'Frogeye Leaf Spot', 'Healty', 'Mossaic Virus', 'Rust', 'Southern blight', 'Sudden Death Syndrome', 'Target Leaf Spot', 'Yellow Mosaic', 'brown_spot', 'crestamento', 'ferrugen', 'powdery_mildew', 'septoria']  # replace with your class names

# --------------------------
# 2. Load trained model
# --------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(1280, len(classes))
model.load_state_dict(torch.load("/content/drive/MyDrive/soybean_model.pth", map_location=device))
model.to(device)
model.eval()

# --------------------------
# 3. Image transforms
# --------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# --------------------------
# 4. Prediction function
# --------------------------
def predict(image):
    # Convert to PIL image
    image = Image.fromarray(image)

    # Apply transforms
    img = transform(image).unsqueeze(0).to(device)

    # Model inference
    with torch.no_grad():
        outputs = model(img)
        _, predicted = torch.max(outputs, 1)

    # Return predicted class
    return classes[predicted.item()]

# --------------------------
# 5. Gradio Interface
# --------------------------
app = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="Upload Leaf Image"),
    outputs=gr.Label(label="Predicted Disease"),
    title=" Leaf Disease Detector",
    description="Upload a oilseed leaf image to detect the disease using a trained EfficientNet model."
)

# Launch the app
app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b41df19a64057af8e2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
